# Calculating hosting capacity for PG&E, Alameda County

Use the hosting capacity calculation methodology from Brockway et al (2021) to calculate the hosting capacity for PG&E at the feederline level.

In order to confirm that our calculations match Brockway's, we will use one specific circuit (ALTO 1122) to compare whether our values are in the same order of magnitude.

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from shapely.geometry import MultiLineString
import matplotlib.pyplot as plt

In [2]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

## Read in Data

### PG&E Utility

In [4]:
# read in feederline data
pge_circuits = gpd.read_file("../../../../../capstone/electrigrid/data/utilities/pge_shapefiles/LineDetail/LineDetail.shp").to_crs("EPSG:4326")

# load in shapefile for extent of PG&E
utility_ter = gpd.read_file("../../../../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

In [5]:
# select circuit of interest
pge_circuits = pge_circuits[pge_circuits['FeederName'] == "ALTO 1122"]

#### Convert `GenCapacit` to MW 

It is currently in KW. We convert in order to standardize to the units of the other two utilities (SDG&E and SCE).

In [6]:
pge_circuits['GenCapacit'] = pge_circuits['GenCapacit'] / 1000
pge_circuits['GenCapac_1'] = pge_circuits['GenCapac_1'] / 1000

#### Brief Exploration

Operational flex allows grid operators to better manage variability and therefore commit more capacity. 

The column that we think is **generation capacity without operational flex** (reverse flow *is* allowed).

In [ ]:
pge_circuits['GenCapacit'].describe()

In [ ]:
pge_circuits['GenCapacit'].plot(kind = "hist")

The column we think is **generation capacity with operational flex** (reverse flow *is not* allowed).

In [ ]:
pge_circuits['GenCapac_1'].describe()

In [ ]:
pge_circuits['GenCapac_1'].plot(kind = "hist")

**Findings:** `GenCapac_1` has lower generation capacity values by a magnitude of 100.

#### Change column names to match other utilites

In [7]:
pge_circuits = pge_circuits.rename(columns = {'FeederId':'circuit_id',
                                              'CSV_LineSe':'segment_id'})

In [ ]:
pge_circuits.head()

### Building

In [9]:
# read in multifamily data
buildings = gpd.read_parquet("../../../../../../capstone/electrigrid/data/building_zillow_merges/marin.parquet").set_crs("EPSG:4326")

We are currently operating with data just for Alameda County, which is under PG&E's jurisdiction. Therefore, instead of clipping to PG&E territory we will clip to the county, rendering the PG&E extent shapefile unnecessary (but only for now!).

#### Clip PG&E utility data to Alameda county

In [ ]:
# create bounding box from building data (which is already clipped to Alameda)
alameda_county_box = box(*alameda_singlefamily.total_bounds)

# clip circuits to that bounding box
pge_alameda = gpd.clip(pge_circuits, alameda_county_box)

### Census

In [10]:
# read in the census tract data
census_tracts = gpd.read_file("../../../../../capstone/electrigrid/data/census/tl_2025_06_tract/tl_2025_06_tract.shp").to_crs("EPSG:4326")

#### Clip census tracts to Alameda county

In [ ]:
census_alameda = gpd.clip(census_tracts, alameda_county_box)

## Preview data

Multi- and single-family homes in blue, feederlines in green, and census tract delineations in gray.

In [ ]:
# plot all of the data sources to ensure everything looks accurate 
fig, ax = plt.subplots(figsize=(10, 10))

ax.axis('off')

census_alameda.boundary.plot(ax=ax, 
                    color='gray')

alameda_singlefamily.plot(ax=ax, 
                    color='blue')

alameda_multifamily.plot(ax=ax, 
                    color='blue')

pge_alameda.plot(ax=ax, 
                      color='green')

## Link Buildings to Feederlines

### Adding length column

Before joining the data frames and replacing the feederline geometries with those of the building, we will calculate the length of the lines and bring them over as a column in the join.

In [11]:
# change the crs to a projected CRS
pge_circuits = pge_circuits.to_crs("EPSG:3310")
buildings = buildings.to_crs("EPSG:3310")

In [12]:
## create length column in metres
pge_circuits['length_m'] = pge_circuits.length

#### Join homes to circuits

In [13]:
# index the data
pge_circuits.sindex
buildings.sindex

# spatial join
linked = gpd.sjoin_nearest(buildings, 
                               pge_circuits, 
                               how="left", 
                               lsuffix='_left', 
                               rsuffix='_right',
                               distance_col='dist_to_line_m')

### Filter for homes that are less than 1000m away from the nearest feederline
If they are more than 1 km away, we assume they get their power from a different source.

In [14]:
# filter
linked = linked[linked['dist_to_line_m'] >= 1000]

In [15]:
linked.head()

,type,year,room,heat,cool,own,value,sqft_type,ID,GEOID,p_ID,code,area_m2,volume_m3,unit,geometry,index__right,circuit_id,FeederName,globalid,segment_id,LoadCapaci,voltage_kv,phase_cnt,limiting_m,limiting_c,ICA_Analys,lica_analy,Division,GenCapacit,GenericPVC,GenCapac_1,GenericCap,limiting_1,limiting_2,limiting_3,limiting_4,limiting_5,limiting_6,limiting_7,limiting_8,ScreenL,Publish,Last_Updat,SHAPE_Leng,length_m,dist_to_line_m
0,Multi,2003.0,1.0,None,None,I,491943.0,living,3,06001403302,468,RR106,4953.551609,122011.206244,224.0,POINT (-199426.975 -22302.163),71921,042031122,ALTO 1122,{EAA178E4-BB52-40F6-B6D6-CBC6105F240A},3731363,750,12.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.91,1150,1.1,1350,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,53.605245,53.549634,25130.442978
1,Multi,2003.0,1.0,None,None,None,240117.0,living,4,06001403302,468,RR106,4953.551609,122011.206244,224.0,POINT (-199426.975 -22302.163),71921,042031122,ALTO 1122,{EAA178E4-BB52-40F6-B6D6-CBC6105F240A},3731363,750,12.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.91,1150,1.1,1350,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,53.605245,53.549634,25130.442978
2,Multi,2003.0,1.0,None,None,I,261770.0,living,5,06001403302,468,RR106,4953.551609,122011.206244,224.0,POINT (-199426.975 -22302.163),71921,042031122,ALTO 1122,{EAA178E4-BB52-40F6-B6D6-CBC6105F240A},3731363,750,12.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.91,1150,1.1,1350,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,53.605245,53.549634,25130.442978
3,Multi,2003.0,1.0,None,None,None,223337.0,living,6,06001403302,468,RR106,4953.551609,122011.206244,224.0,POINT (-199426.975 -22302.163),71921,042031122,ALTO 1122,{EAA178E4-BB52-40F6-B6D6-CBC6105F240A},3731363,750,12.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.91,1150,1.1,1350,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,53.605245,53.549634,25130.442978
5,Multi,2003.0,1.0,None,None,None,278288.0,living,8,06001403302,468,RR106,4953.551609,122011.206244,224.0,POINT (-199426.975 -22302.163),71921,042031122,ALTO 1122,{EAA178E4-BB52-40F6-B6D6-CBC6105F240A},3731363,750,12.0,3,None,None,Feb 2025,Feb 2025,Sonoma,0.91,1150,1.1,1350,None,None,None,None,None,None,None,None,Likely to pass,1,2025-06-02,53.605245,53.549634,25130.442978


## Hosting Capacity Calculation

### Step 1: Create length column

Completed before link!

**Note:** `SHAPE_Leng` is the "GIS-provided" length, which is often in decimal degrees – this is why the two columns don't match.

### Step 2: Add census tract ID to each home

In [16]:
## single family join

# change CRS so it matches
linked = linked.to_crs("EPSG:4326")

assert linked.crs == census_tracts.crs

# drop columns that makes joins impossible
linked = linked.drop(['index_right'], axis = 1)

# join
linked = linked.sjoin(
    census_tracts,
    how = "left",
    predicate = "intersects")

KeyError: "['index_right'] not found in axis"

In [ ]:
# join
linked = linked.sjoin(
    census_tracts,
    how = "left",
    predicate = "intersects")

In [ ]:
alameda_singlefamily_linked.head(2)

### Step 3: Calculate the number of homes in each census tract

In [ ]:
# calculate the number of observations in each
by_tract = linked.groupby('GEOID_right').sum('unit')

# save just the unit column and rename to be more intuitive
units_by_tract = by_tract['unit']
units_by_tract = units_by_tract.rename('units_by_tract')

# join units by tract to entire df
linked = linked.merge(
    units_by_tract,
    on = "GEOID_right")

### Step 4: Calculate the maximum ICA generation capacity for each circuit

In [ ]:
# find maximum generation across feederline
max_gen = linked.groupby('circuit_id').max('GenCapacit')

# save just the necessary column and rename
max_gen_feeder = max_gen['GenCapacit'].rename('max_gen')

# join max generation by feeder to entire df
linked = linked.merge(
    max_gen_feeder,
    on = 'circuit_id')

### Step 5: Calculate the percentage of the length of each segment relative to the entire feederline/circuit

In [ ]:
# find the total length by Feeder Id
total_length = linked.groupby('circuit_id').sum('length_m')

# rename
total_length = total_length['length_m'].rename('total_feeder_length')

# join
linked = linked.merge(
    total_length,
    on = 'circuit_id')

In [ ]:
# calculate percentage
linked['perc_length'] = (linked['length_m'] / linked['total_feeder_length']) * 100

In [ ]:
alameda_singlefamily_linked.head()

**Note:** Why are there multiple rows with the same `CSV_LineSe`?

### Step 6: Calculate the number of homes connected to each segment

In [ ]:
# find the number of homes
home_count_seg = linked.groupby('segment_id').sum('unit')

# save just the unit columns and rename to be more intuitive
home_count_seg = home_count_seg['unit'].rename('home_count_seg')

# join with original df based on line segment 
linked = linked.merge(
    home_count_seg,
    on = "segment_id")

### Step 7: Calculate the number of homes connected to each circuit

In [ ]:
# find the number of homes
home_count_seg = linked.groupby('circuit_id').sum('unit')

# save just the unit columns and rename to be more intuitive
home_count_seg = home_count_seg['unit'].rename('home_count_circuit')

# join with original df based on line segment 
linked = linked.merge(
    home_count_seg,
    on = "circuit_id")

### Step 8: Calculate the number of homes connected to segment relative to the number of homes connected to entire feederline/circuit

In [ ]:
# calculate perc of homes and save as column
linked['perc_homes'] = (linked['home_count_seg'] / linked['home_count_circuit']) * 100

### Step 9: Calculate weighted generation capacity for each segment
Multiply the maximum generation capacity of each circuit by the the percentage of households each segment represents relative to the whole (`perc_homes`). This will undercount the total maximum hosting capacity of each circuit.

### Step 10: Calculate the number of homes located within each census tract for each circuit

**Variable:** # of households within circuit polygon

Circuit polygon definition: corresponds to the overlap between census tract and the estimated service area of circuit

In [ ]:
circ_poly = linked.groupby(['GEOID_right', 'circuit_id']).sum('unit')

circ_poly_units = circ_poly['unit'].rename('tothh_Cpoly')

linked = linked.merge(
    circ_poly_units,
    on = ['GEOID_right', 'circuit_id'])

### Step 10b: Calculate the max generation within each census tract for each circuit

**Variable:** maximum generation within census tract, by feederline

In [ ]:
circ_poly_gen = linked.groupby(['GEOID_right', 'circuit_id']).max('GenCapacit')

circ_poly_gen_max = circ_poly['GenCapacit'].rename('DER_max_Cpoly')

linked = linked.merge(
    circ_poly_gen_max,
    on = ['GEOID_right', 'circuit_id'])

### Step 11: Weight by household (complete Equation 8)

**Equation 8 from Brockway et al.**: 

_hhWt = (max capacity across circuit polygon) * (# of households within circuit polygon / # of households served by circuit)

**or**

`_hhWt` = `DER_max_Cpoly` * ( `tothh_Cpoly` / `home_count_circuit` )

In [ ]:
# calculate weight by household
linked['_hhWt'] = linked['DER_max_Cpoly'] * (linked['tothh_Cpoly'] / linked['home_count_circuit'])

**Note:** Values are different for observations with same `CSV_LineSe` but different GEOIDs.

### Step 12: Normalize household weight (complete Equation 9)

**Equation 9 from Brockway et al.:**

_hhWt_n = (household-weight max capacity) * (maximum capacity anywhere on circuit / household-weighted max capacity summed across circuit)

**or**

`_hhWt_n` = `_hhWt` * (`max_gen` / summ j [`_hhWt`] )

In [ ]:
# calculate summation of household-weighted max capacity across circuit
summ_hhWt = linked.groupby('circuit_id').sum('_hhWt')

# save the summed column and rename
summ_hhWt = summ_hhWt['_hhWt'].rename('summ_hhWt')

# add to data frame through join
linked = linked.merge(
    summ_hhWt,
    on = ['circuit_id'])

In [ ]:
# equation 9
linked['_hhWt_n'] = linked['_hhWt'] * (linked['max_gen'] / linked['summ_hhWt'])

### Step 13: Adjust for observations where max capacity is exceeded (complete Equation 10)

"For a small minority of circuit polygons (0.05–1.60%, depending on DER type),
the normalized hosting capacity value exceeds the maximum allowed hosting
capacity in that circuit polygon. In these cases, we adjusted the value back to its
allowed maximum:"

**Equation 10 from Brockway et al.:

_hhWt_n = 

    if _hhWt_n > (max capacity across circuit polygon), then replace with max capacity across circuit polygon,
    else: 
            _hhWt_n
    
**or**

`_hhWt_n` = if `_hhWt_n` > `DER_max_Cpoly`, then `DER_max_Cpoly`,
            else `_hhWt_n`

In [ ]:
linked['_hhWt_n'] = np.where(
    
    # condition -- weighted generation is greater than max generation across circuit/feederline
    linked['_hhWt_n'] > linked['DER_max_Cpoly'],
    
    # replace with max generation of feederline if condition is true
    linked['DER_max_Cpoly'], 
    
    # otherwise, keep original calculated value
    linked['_hhWt_n'])

**Note:** Could potentially check if this worked by creating a completely new column (instead of just replacing the old one), and seeing how the two compare.

### Step 14: Calculate remaining per-household capacity

**Equation 11 from Brockway et al.:**

`DER_remain`, or kW per household = `_hhWt_n` / `tothh_Cpoly`

In [ ]:
linked['DER_remain'] = linked['_hhWt_n'] / linked['tothh_Cpoly'] * 1000

### Step 15: Calculate final per-household hosting capacity

**Equation 12 from Brockway et al.**

`DER` = `DER_exist` + `DER_remain`

In [ ]:
linked_clean = linked[['ID', 'unit', 'bbox', 'geometry', # columns used in analysis
                                                                        'circuit_id', 'segment_id', 'GenCapacit', 
                                                                        'dist_to_line_m','GEOID_right', 
                                                                        
                                                                        # columns created for analysis
                                                                        'length_m', 'units_by_tract', 'max_gen', 'total_feeder_length', 'perc_length',
                                                                        'home_count_seg', 'home_count_circuit', 'perc_homes',
                                                                        'tothh_Cpoly', 'DER_max_Cpoly', '_hhWt', '_hhWt_n', 'DER_remain']]

In [ ]:
linked_clean.head()

## GEOID check

In [ ]:
alameda_singlefamily_linked['GEOID_right'] == alameda_singlefamily_linked['GEOID_left']

In [ ]:
alameda_singlefamily_linked['GEOID_right'].eq(alameda_singlefamily_linked['GEOID_left']).value_counts()

In [ ]:
alameda_singlefamily_linked.head()

In [ ]:
alameda_singlefamily_linked['DER_remain'].describe()

In [ ]:
alameda_singlefamily_linked['DER_remain'].plot(kind = "hist",
                                              xlim=(0, 500),
                                              bins=range(0, 500, 25))

**Note:** Without Op Flex, no negatives! 

## Repeat workflow for generation capacity *with* operational flex

Starting with Step 4 --> the first place generation capacity is introduced.

In [ ]:
# replicate entire pipeline from above

# Step 4
# find maximum generation across feederline
max_gen = alameda_singlefamily_linked.groupby('circuit_id').max('GenCapac_1')

# save just the necessary column and rename
max_gen_feeder = max_gen['GenCapac_1'].rename('max_gen_opflex')

# join max generation by feeder to entire df
alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    max_gen_feeder,
    on = 'circuit_id')

# Step 10b
circ_poly_gen = alameda_singlefamily_linked.groupby(['GEOID_right', 'circuit_id']).max('GenCapac_1')

circ_poly_gen_max = circ_poly['GenCapac_1'].rename('DER_max_Cpoly_opflex')

alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    circ_poly_gen_max,
    on = ['GEOID_right', 'circuit_id'])

# Step 11
# calculate weight by household
alameda_singlefamily_linked['_hhWt_opflex'] = alameda_singlefamily_linked['DER_max_Cpoly_opflex'] * (alameda_singlefamily_linked['tothh_Cpoly'] / alameda_singlefamily_linked['home_count_circuit'])

# Step 12
# calculate summation of household-weighted max capacity across circuit
summ_hhWt = alameda_singlefamily_linked.groupby('circuit_id').sum('_hhWt_opflex')

# save the summed column and rename
summ_hhWt = summ_hhWt['_hhWt_opflex'].rename('summ_hhWt_opflex')

# add to data frame through join
alameda_singlefamily_linked = alameda_singlefamily_linked.merge(
    summ_hhWt,
    on = ['circuit_id'])

# equation 9
alameda_singlefamily_linked['_hhWt_n_opflex'] = alameda_singlefamily_linked['_hhWt_opflex'] * (alameda_singlefamily_linked['max_gen_opflex'] / alameda_singlefamily_linked['summ_hhWt_opflex'])

# Step 13
alameda_singlefamily_linked['_hhWt_n_opflex'] = np.where(
    
    # condition -- weighted generation is greater than max generation across circuit/feederline
    alameda_singlefamily_linked['_hhWt_n_opflex'] > alameda_singlefamily_linked['DER_max_Cpoly_opflex'],
    
    # replace with max generation of feederline if condition is true
    alameda_singlefamily_linked['DER_max_Cpoly_opflex'], 
    
    # otherwise, keep original calculated value
    alameda_singlefamily_linked['_hhWt_n_opflex'])

# Step 14
alameda_singlefamily_linked['DER_remain_opflex'] = alameda_singlefamily_linked['_hhWt_n_opflex'] / alameda_singlefamily_linked['tothh_Cpoly'] * 1000

#### Explore

In [ ]:
alameda_singlefamily_linked['DER_remain_opflex'].describe()

In [ ]:
alameda_singlefamily_linked['DER_remain'].describe()

**With** operational flex appears to have larger generation capacity values, and no negative ones (opposite of the raw exploration we did before...does this make sense?)

In [ ]:
alameda_singlefamily_linked['DER_remain_opflex'].plot(kind = "hist")

minor change